# LSApp Time-Diff Preparation
This notebook loads LSApp event logs, computes interaction/session IDs, derives `open_time` and `close_time`, and saves processed outputs into `data/with_time_diff`.

In [1]:
%matplotlib inline

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)

In [2]:
# Fixed paths (no candidate search).
project_root = Path.cwd().parent if Path.cwd().name == 'src' else Path.cwd()
input_path = (project_root / 'data' / 'lsapp.tsv' / 'lsapp.tsv').resolve()

if not input_path.exists():
    raise FileNotFoundError(f'Could not find input file: {input_path}')

output_dir = (project_root / 'data' / 'with_time_diff').resolve()
output_dir.mkdir(parents=True, exist_ok=True)

print(f'Input file: {input_path}')
print(f'Output dir: {output_dir}')

Input file: D:\Git\context-aware-user-recommendation\data\lsapp.tsv\lsapp.tsv
Output dir: D:\Git\context-aware-user-recommendation\data\with_time_diff


In [3]:
df = pd.read_csv(input_path, sep='\t')
df = df.rename(columns={'lsapp.tsv': 'user_id'})

print(df.shape)
df.head()

(3658590, 5)


,user_id,session_id,timestamp,app_name,event_type
0,0.0,1.0,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened
1,0.0,1.0,2018-01-16 06:01:05,Minesweeper Classic (Mines),Closed
2,0.0,1.0,2018-01-16 06:01:07,Minesweeper Classic (Mines),Opened
3,0.0,1.0,2018-01-16 06:01:07,Minesweeper Classic (Mines),Closed
4,0.0,1.0,2018-01-16 06:01:08,Minesweeper Classic (Mines),Opened


In [4]:
# Basic type cleanup and ordering before ID construction.
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp', 'user_id', 'app_name']).copy()
df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)

# Optional helper subset for open events.
df_open = df.loc[df['event_type'] == 'Opened'].copy()

print('Clean rows:', len(df))
print('Opened rows:', len(df_open))

Clean rows: 3658589
Opened rows: 1673261


In [5]:
# Reproduce interaction/session logic similar to your reference notebook.
df['time_dff'] = df['timestamp'].diff()

is_new_interaction = (
    ((df['timestamp'] - df['timestamp'].shift(1) > pd.Timedelta(1, 'm')) & (df['event_type'] == 'Opened'))
    | ~(df['app_name'] == df['app_name'].shift(1))
    | ~(df['user_id'] == df['user_id'].shift(1))
)

is_new_session = (
    ((df['timestamp'] - df['timestamp'].shift(1) > pd.Timedelta(5, 'm')) & (df['event_type'] == 'Opened'))
    | ~(df['user_id'] == df['user_id'].shift(1))
)

df['interaction_id'] = is_new_interaction.cumsum()
df['session_id'] = is_new_session.cumsum()

In [6]:
# Create one row per interaction (same style as your reference notebook).
df_start = df.drop_duplicates(subset=['interaction_id'], keep='first').copy()
df_end = df.drop_duplicates(subset=['interaction_id'], keep='last').copy()

df_start = df_start.set_index('interaction_id')
df_end = df_end.set_index('interaction_id')

df_start['open_time'] = df_start['timestamp']
df_start['close_time'] = df_end['timestamp']
df_start['duration(s)'] = (df_start['close_time'] - df_start['open_time']).dt.total_seconds()

# Keep interaction-level dataset only (no repeated event rows).
df_interaction = df_start.reset_index().copy()

df_interaction.head(15)

,interaction_id,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time,duration(s)
0,1,0.0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaT,2018-01-16 06:01:05,2018-01-16 06:01:09,4.0
1,2,0.0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17,33.0
2,3,0.0,1,2018-01-16 06:25:54,Gmail,User Interaction,0 days 00:21:37,2018-01-16 06:25:54,2018-01-16 06:25:54,0.0
3,4,0.0,1,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10,5.0
4,5,0.0,1,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21,11.0
5,6,0.0,1,2018-01-16 06:26:21,Google,Opened,0 days 00:00:00,2018-01-16 06:26:21,2018-01-16 06:26:26,5.0
6,7,0.0,1,2018-01-16 06:26:26,Google Chrome,Opened,0 days 00:00:00,2018-01-16 06:26:26,2018-01-16 06:36:26,600.0
7,8,0.0,2,2018-01-16 07:15:56,Minesweeper Classic (Mines),Opened,0 days 00:39:30,2018-01-16 07:15:56,2018-01-16 07:15:58,2.0
8,9,0.0,2,2018-01-16 07:20:45,Google Chrome,Opened,0 days 00:04:47,2018-01-16 07:20:45,2018-01-16 07:20:45,0.0
9,10,0.0,2,2018-01-16 07:20:46,Minesweeper Classic (Mines),Opened,0 days 00:00:01,2018-01-16 07:20:46,2018-01-16 07:21:43,57.0


In [7]:
# Filter out very short interactions (< 3 seconds).
df_interaction = df_interaction[df_interaction['duration(s)'] >= 3].copy()

print('Sessions after duration filter:', df_interaction['session_id'].nunique())
print('Rows after duration filter:', len(df_interaction))
df_interaction

Sessions after duration filter: 31376
Rows after duration filter: 209156


,interaction_id,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time,duration(s)
0,1,0.0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaT,2018-01-16 06:01:05,2018-01-16 06:01:09,4.0
1,2,0.0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17,33.0
3,4,0.0,1,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10,5.0
4,5,0.0,1,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21,11.0
5,6,0.0,1,2018-01-16 06:26:21,Google,Opened,0 days 00:00:00,2018-01-16 06:26:21,2018-01-16 06:26:26,5.0
...,...,...,...,...,...,...,...,...,...,...
599625,599626,291.0,33217,2018-04-06 12:05:33,Facebook,Opened,0 days 00:00:00,2018-04-06 12:05:33,2018-04-06 12:05:50,17.0
599626,599627,291.0,33217,2018-04-06 12:05:50,Facebook Messenger,Opened,0 days 00:00:00,2018-04-06 12:05:50,2018-04-06 12:18:51,781.0
599629,599630,291.0,33218,2018-04-06 12:37:57,Facebook Messenger,Opened,0 days 00:13:42,2018-04-06 12:37:57,2018-04-06 12:43:59,362.0
599630,599631,291.0,33218,2018-04-06 12:47:28,Facebook Messenger,Opened,0 days 00:03:29,2018-04-06 12:47:28,2018-04-06 12:53:13,345.0


In [8]:
# Unique app_name list and count, then save to output/UniqueAppNames.
unique_apps = pd.Series(df['app_name'].dropna().astype(str).unique()).sort_values().reset_index(drop=True)
print('Number of unique app_name:', len(unique_apps))

unique_app_dir = project_root / 'output' / 'UniqueAppNames'
unique_app_dir.mkdir(parents=True, exist_ok=True)

txt_out = unique_app_dir / 'unique_app_names.txt'

unique_apps.to_csv(txt_out, index=False, header=False)

print(f'Saved txt: {txt_out}')

# Preview
unique_apps.head(30)

Number of unique app_name: 87
Saved txt: d:\Git\context-aware-user-recommendation\output\UniqueAppNames\unique_app_names.txt


0                         AOL
1             Amazon Shopping
2          Android In Call UI
3             Army Men Strike
4                       Badoo
5               Baseball Boy!
6               Brave Browser
7                  Calculator
8                    Calendar
9             Calorie Counter
10                     Camera
11               Clean Master
12                      Clock
13                   Contacts
14    DigiHUD Pro Speedometer
15                    Discord
16                EntertaiNow
17                   Facebook
18         Facebook Messenger
19                      Faceu
20                     Flickr
21         Flipboard Briefing
22                      Gmail
23                     Google
24              Google Chrome
25               Google Drive
26              Google Photos
27          Google Play Music
28          Google Play Store
29                   Hangouts
dtype: str

In [9]:
# Add app categories to the filtered df_interaction (no separate output file).
mapping_path = project_root / 'output' / 'UniqueAppNames' / 'app_category_mapping_87.txt'

if not mapping_path.exists():
    raise FileNotFoundError(f'Mapping file not found: {mapping_path}')

# 1) Read mapping as tab-separated file with required columns.
mapping_df = pd.read_csv(mapping_path, sep='\t')
expected_cols = {'app_name', 'app_category'}
if set(mapping_df.columns) != expected_cols:
    raise ValueError(
        f'Mapping file must have exactly columns {expected_cols}, got: {list(mapping_df.columns)}'
    )

# 2-4) Merge mapping by exact app_name and keep all original columns.
df_interaction = df_interaction.merge(
    mapping_df,
    on='app_name',
    how='left',
    validate='many_to_one'
).copy()

# 5) Print required checks.
num_rows = len(df_interaction)
num_unique_apps = df_interaction['app_name'].nunique()
num_missing_category = int(df_interaction['app_category'].isna().sum())
missing_apps = sorted(df_interaction.loc[df_interaction['app_category'].isna(), 'app_name'].dropna().unique().tolist())

print('Rows:', num_rows)
print('Unique apps:', num_unique_apps)
print('Missing app_category values:', num_missing_category)
print('App names missing app_category:', missing_apps if missing_apps else 'None')

# 6) Stop if anything is missing.
if num_missing_category > 0:
    raise RuntimeError('Missing app_category detected. Please update app_category_mapping_87.txt before saving.')

df_interaction.head(10)

Rows: 209156
Unique apps: 87
Missing app_category values: 0
App names missing app_category: None


,interaction_id,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time,duration(s),app_category
0,1,0.0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaT,2018-01-16 06:01:05,2018-01-16 06:01:09,4.0,Games
1,2,0.0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17,33.0,Games
2,4,0.0,1,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10,5.0,Browser_Search
3,5,0.0,1,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21,11.0,Social
4,6,0.0,1,2018-01-16 06:26:21,Google,Opened,0 days 00:00:00,2018-01-16 06:26:21,2018-01-16 06:26:26,5.0,Browser_Search
5,7,0.0,1,2018-01-16 06:26:26,Google Chrome,Opened,0 days 00:00:00,2018-01-16 06:26:26,2018-01-16 06:36:26,600.0,Browser_Search
6,10,0.0,2,2018-01-16 07:20:46,Minesweeper Classic (Mines),Opened,0 days 00:00:01,2018-01-16 07:20:46,2018-01-16 07:21:43,57.0,Games
7,14,0.0,3,2018-01-16 08:03:34,Minesweeper Classic (Mines),Opened,0 days 00:00:00,2018-01-16 08:03:34,2018-01-16 08:04:10,36.0,Games
8,18,0.0,4,2018-01-16 08:31:51,Minesweeper Classic (Mines),Opened,0 days 00:00:01,2018-01-16 08:31:51,2018-01-16 08:34:13,142.0,Games
9,22,0.0,5,2018-01-16 09:47:53,Minesweeper Classic (Mines),Opened,0 days 00:00:01,2018-01-16 09:47:53,2018-01-16 09:48:23,30.0,Games


In [10]:
interaction_out = output_dir / 'lsapp_interactions_with_time_diff_filtered.tsv'

df_interaction.to_csv(interaction_out, sep='\t', index=False)

print(f'Saved filtered interaction file (with app_category): {interaction_out}')
print('interaction rows:', len(df_interaction))

Saved filtered interaction file (with app_category): D:\Git\context-aware-user-recommendation\data\with_time_diff\lsapp_interactions_with_time_diff_filtered.tsv
interaction rows: 209156


## User-Day Feature Engineering
Build a clean user-day-level feature table from the interaction-level LSApp data.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def load_data(project_root: Path) -> pd.DataFrame:
    """Load interaction-level LSApp dataset used for user-day feature engineering."""
    input_file = project_root / 'data' / 'with_time_diff' / 'lsapp_interactions_with_time_diff_filtered.tsv'
    if not input_file.exists():
        raise FileNotFoundError(f'Input dataset not found: {input_file}')

    df = pd.read_csv(input_file, sep='\t')
    required_cols = {
        'user_id',
        'session_id',
        'app_name',
        'app_category',
        'open_time',
        'close_time',
        'duration(s)',
    }
    missing = sorted(required_cols.difference(df.columns))
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    print(f'Loaded interaction dataset: {input_file}')
    print(f'Raw shape: {df.shape}')
    return df


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Parse datetimes, build date field, and clean invalid duration values."""
    cleaned = df.copy()

    cleaned['open_time'] = pd.to_datetime(cleaned['open_time'], errors='coerce')
    cleaned['close_time'] = pd.to_datetime(cleaned['close_time'], errors='coerce')
    cleaned['date'] = cleaned['open_time'].dt.date

    datetime_missing_mask = cleaned[['open_time', 'close_time', 'date']].isna().any(axis=1)
    cleaned = cleaned.loc[~datetime_missing_mask].copy()
    print(f'Removed rows with invalid datetime/date fields: {int(datetime_missing_mask.sum())}')

    cleaned['duration(s)'] = cleaned['duration(s)'].astype(float)
    cleaned['app_name'] = cleaned['app_name'].fillna('unknown_app').astype(str).str.strip()
    cleaned['app_category'] = cleaned['app_category'].fillna('unknown').astype(str).str.strip()
    cleaned.loc[cleaned['app_name'] == '', 'app_name'] = 'unknown_app'
    cleaned.loc[cleaned['app_category'] == '', 'app_category'] = 'unknown'

    cleaned = cleaned.sort_values(['user_id', 'date', 'open_time', 'close_time']).reset_index(drop=True)

    return cleaned


def compute_entropy(duration_values: pd.Series) -> float:
    """Compute entropy from duration proportions (base-2), returning 0 for undefined cases."""
    values = pd.Series(duration_values, dtype='float64')
    values = values[values > 0]
    total = values.sum()
    if total <= 0:
        return 0.0

    proportions = values / total
    return float(-(proportions * np.log2(proportions)).sum())


def _clean_category_name(category: str) -> str:
    normalized = ''.join(ch.lower() if ch.isalnum() else '_' for ch in str(category).strip())
    normalized = '_'.join(part for part in normalized.split('_') if part)
    return normalized if normalized else 'unknown'


def build_user_day_features(df: pd.DataFrame) -> pd.DataFrame:
    """Build user-day-level features from cleaned interaction-level data."""
    group_cols = ['user_id', 'date']

    # Usage intensity features.
    intensity = (
        df.groupby(group_cols)
        .agg(
            total_duration=('duration(s)', 'sum'),
            total_interactions=('duration(s)', 'size'),
            num_sessions=('session_id', pd.Series.nunique),
            avg_interaction_duration=('duration(s)', 'mean'),
            median_interaction_duration=('duration(s)', 'median'),
            max_interaction_duration=('duration(s)', 'max'),
        )
        .reset_index()
    )

    session_duration = (
        df.groupby(group_cols + ['session_id'], as_index=False)['duration(s)']
        .sum()
        .rename(columns={'duration(s)': 'session_duration'})
    )
    avg_session_duration = (
        session_duration.groupby(group_cols, as_index=False)['session_duration']
        .mean()
        .rename(columns={'session_duration': 'avg_session_duration'})
    )

    # Diversity features (entropy from duration proportions).
    diversity = (
        df.groupby(group_cols)
        .agg(
            num_unique_apps=('app_name', pd.Series.nunique),
            num_unique_categories=('app_category', pd.Series.nunique),
        )
        .reset_index()
    )

    app_duration = df.groupby(group_cols + ['app_name'], as_index=False)['duration(s)'].sum()
    app_entropy = (
        app_duration.groupby(group_cols)['duration(s)']
        .apply(compute_entropy)
        .reset_index(name='app_entropy')
    )

    category_duration = df.groupby(group_cols + ['app_category'], as_index=False)['duration(s)'].sum()
    category_entropy = (
        category_duration.groupby(group_cols)['duration(s)']
        .apply(compute_entropy)
        .reset_index(name='category_entropy')
    )

    # Category duration and ratio features.
    category_pivot = category_duration.pivot_table(
        index=group_cols,
        columns='app_category',
        values='duration(s)',
        fill_value=0.0,
        aggfunc='sum',
    )
    category_pivot = category_pivot.rename(columns=lambda c: f"{_clean_category_name(c)}_duration")
    category_pivot = category_pivot.reset_index()

    # Temporal bucket ratio features based on open_time.
    hour = df['open_time'].dt.hour
    conditions = [
        (hour >= 6) & (hour <= 11),
        (hour >= 12) & (hour <= 17),
        (hour >= 18) & (hour <= 23),
    ]
    choices = ['morning', 'afternoon', 'evening']
    df_temp = df.copy()
    df_temp['time_bucket'] = np.select(conditions, choices, default='late_night')

    bucket_duration = (
        df_temp.groupby(group_cols + ['time_bucket'], as_index=False)['duration(s)']
        .sum()
        .rename(columns={'duration(s)': 'bucket_duration'})
    )
    bucket_pivot = bucket_duration.pivot_table(
        index=group_cols,
        columns='time_bucket',
        values='bucket_duration',
        fill_value=0.0,
        aggfunc='sum',
    ).reset_index()

    # Behavioral dynamics features from sorted interactions.
    ordered = df.sort_values(group_cols + ['open_time', 'close_time']).copy()
    ordered['prev_app'] = ordered.groupby(group_cols)['app_name'].shift(1)
    ordered['prev_category'] = ordered.groupby(group_cols)['app_category'].shift(1)

    ordered['app_switch'] = ((ordered['app_name'] != ordered['prev_app']) & ordered['prev_app'].notna()).astype(int)
    ordered['category_switch'] = ((ordered['app_category'] != ordered['prev_category']) & ordered['prev_category'].notna()).astype(int)
    ordered['gap_seconds'] = ordered.groupby(group_cols)['open_time'].diff().dt.total_seconds()

    behavior = (
        ordered.groupby(group_cols)
        .agg(
            app_switch_count=('app_switch', 'sum'),
            category_switch_count=('category_switch', 'sum'),
            avg_gap_between_interactions_seconds=('gap_seconds', 'mean'),
        )
        .reset_index()
    )

    # Date context features.
    context = intensity[group_cols].copy()
    date_dt = pd.to_datetime(context['date'])
    context['day_of_week'] = date_dt.dt.day_name().str.lower()
    context['is_weekend'] = context['day_of_week'].isin(['saturday', 'sunday']).astype(int)

    features = intensity.merge(avg_session_duration, on=group_cols, how='left')
    features = features.merge(diversity, on=group_cols, how='left')
    features = features.merge(app_entropy, on=group_cols, how='left')
    features = features.merge(category_entropy, on=group_cols, how='left')
    features = features.merge(category_pivot, on=group_cols, how='left')
    features = features.merge(bucket_pivot, on=group_cols, how='left')
    features = features.merge(behavior, on=group_cols, how='left')
    features = features.merge(context, on=group_cols, how='left')

    # Ratio features for categories and time-of-day buckets.
    duration_columns = [c for c in features.columns if c.endswith('_duration') and c not in {
        'total_duration',
        'avg_interaction_duration',
        'median_interaction_duration',
        'max_interaction_duration',
        'avg_session_duration',
    }]
    category_duration_columns = [
        c for c in duration_columns
        if c not in {'morning', 'afternoon', 'evening', 'late_night'}
    ]

    for col in category_duration_columns:
        ratio_col = col.replace('_duration', '_ratio')
        features[ratio_col] = np.where(features['total_duration'] > 0, features[col] / features['total_duration'], 0.0)

    for bucket in ['morning', 'afternoon', 'evening', 'late_night']:
        if bucket not in features.columns:
            features[bucket] = 0.0
        ratio_col = f'{bucket}_duration_ratio'
        features[ratio_col] = np.where(features['total_duration'] > 0, features[bucket] / features['total_duration'], 0.0)

    features['switch_rate_per_interaction'] = np.where(
        features['total_interactions'] > 0,
        features['app_switch_count'] / features['total_interactions'],
        0.0,
    )

    # Missing value handling requested by specification.
    ratio_cols = [c for c in features.columns if c.endswith('_ratio')]
    features[ratio_cols] = features[ratio_cols].fillna(0.0)
    features['app_entropy'] = features['app_entropy'].fillna(0.0)
    features['category_entropy'] = features['category_entropy'].fillna(0.0)
    features['avg_gap_between_interactions_seconds'] = features['avg_gap_between_interactions_seconds'].fillna(0.0)

    for col in ['app_switch_count', 'category_switch_count', 'num_sessions', 'total_interactions']:
        features[col] = features[col].fillna(0).astype(int)

    # Keep date as date-only string for TSV output and readability.
    features['date'] = pd.to_datetime(features['date']).dt.date

    # Drop temporary bucket duration columns, keep only required ratio features.
    features = features.drop(columns=['morning', 'afternoon', 'evening', 'late_night'], errors='ignore')

    features = features.sort_values(group_cols).reset_index(drop=True)
    return features


def save_features(features: pd.DataFrame, project_root: Path) -> Path:
    """Save user-day feature table to TSV under data/features."""
    output_dir = project_root / 'data' / 'features'
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / 'lsapp_user_day_features.tsv'
    features.to_csv(output_file, sep='\t', index=False)
    return output_file


# Run full user-day feature pipeline.
project_root = Path.cwd().parent if Path.cwd().name == 'src' else Path.cwd()
raw_df = load_data(project_root)
clean_df = clean_data(raw_df)
user_day_features = build_user_day_features(clean_df)
output_file = save_features(user_day_features, project_root)

# Quality checks.
print('\n=== Quality Checks ===')
print('Final feature table shape:', user_day_features.shape)
print('Number of users:', user_day_features['user_id'].nunique())
print('Number of dates:', user_day_features['date'].nunique())
print('Number of user-day rows:', len(user_day_features))
print('\nFeature columns:')
print(user_day_features.columns.tolist())
print('\nMissing value count per column:')
print(user_day_features.isna().sum())

key_features = [
    'total_duration',
    'total_interactions',
    'num_sessions',
    'avg_interaction_duration',
    'avg_session_duration',
    'num_unique_apps',
    'num_unique_categories',
    'app_entropy',
    'category_entropy',
    'app_switch_count',
    'category_switch_count',
    'switch_rate_per_interaction',
    'avg_gap_between_interactions_seconds',
]
key_features = [col for col in key_features if col in user_day_features.columns]
print('\nSummary statistics for key features:')
print(user_day_features[key_features].describe().T)

print(f'\nSaved user-day feature table: {output_file}')

Loaded interaction dataset: d:\Git\context-aware-user-recommendation\data\with_time_diff\lsapp_interactions_with_time_diff_filtered.tsv
Raw shape: (209156, 11)
Removed rows with invalid datetime/date fields: 0

=== Quality Checks ===
Final feature table shape: (2907, 87)
Number of users: 292
Number of dates: 239
Number of user-day rows: 2907

Feature columns:
['user_id', 'date', 'total_duration', 'total_interactions', 'num_sessions', 'avg_interaction_duration', 'median_interaction_duration', 'max_interaction_duration', 'avg_session_duration', 'num_unique_apps', 'num_unique_categories', 'app_entropy', 'category_entropy', 'app_store_duration', 'browser_search_duration', 'communication_duration', 'communication_email_duration', 'communication_social_duration', 'entertainment_duration', 'finance_duration', 'games_duration', 'health_fitness_duration', 'maps_navigation_duration', 'music_audio_duration', 'news_duration', 'news_search_duration', 'news_utility_duration', 'photo_video_duration',

## Clustering Feature Post-Processing
Apply sparsity check for category ratio columns, encode weekday as numeric, log-transform skewed gap feature, and build clustering-ready features.

In [7]:
import re


def merge_sparse_category_ratio_features(df: pd.DataFrame, zero_threshold: float = 0.95) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Merge sparse sub-category ratio columns back to parent category ratio columns.
    Example: social_dating_ratio -> social_ratio when sparse.
    """
    out = df.copy()

    excluded_ratio_cols = {
        'morning_duration_ratio',
        'afternoon_duration_ratio',
        'evening_duration_ratio',
        'late_night_duration_ratio',
        'switch_rate_per_interaction',
    }

    ratio_cols = [
        c for c in out.columns
        if c.endswith('_ratio') and c not in excluded_ratio_cols
    ]

    sparse_report = []
    pattern = re.compile(r'^(?P<parent>[a-z0-9]+)_[a-z0-9_]+_ratio$')

    for col in ratio_cols:
        zero_ratio = float((out[col] == 0).mean())
        is_sparse = zero_ratio >= zero_threshold

        if not is_sparse:
            continue

        match = pattern.match(col)
        if not match:
            continue

        parent_col = f"{match.group('parent')}_ratio"
        if parent_col == col:
            continue

        if parent_col not in out.columns:
            out[parent_col] = 0.0

        out[parent_col] = out[parent_col].fillna(0.0) + out[col].fillna(0.0)
        sparse_report.append(
            {
                'sparse_col': col,
                'parent_col': parent_col,
                'zero_ratio': zero_ratio,
                'action': 'merged_to_parent',
            }
        )
        out = out.drop(columns=[col])

    report_df = pd.DataFrame(sparse_report).sort_values('zero_ratio', ascending=False) if sparse_report else pd.DataFrame(
        columns=['sparse_col', 'parent_col', 'zero_ratio', 'action']
    )

    return out, report_df


def build_clustering_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create clustering-oriented feature set based on selection rules."""
    out = df.copy()

    # Convert day_of_week to numeric Monday=0 ... Sunday=6.
    dow_map = {
        'monday': 0,
        'tuesday': 1,
        'wednesday': 2,
        'thursday': 3,
        'friday': 4,
        'saturday': 5,
        'sunday': 6,
    }
    out['day_of_week_num'] = out['day_of_week'].astype(str).str.lower().map(dow_map)

    # Log-transform heavily skewed average gap.
    out['avg_gap_between_interactions_seconds_log1p'] = np.log1p(
        out['avg_gap_between_interactions_seconds'].clip(lower=0)
    )

    # Keep: ratio, entropy, switch, time-of-day ratio, unique counts.
    ratio_cols = [
        c for c in out.columns
        if c.endswith('_ratio') and c != 'switch_rate_per_interaction'
    ]
    entropy_cols = [c for c in out.columns if 'entropy' in c]
    switch_cols = [
        c for c in [
            'app_switch_count',
            'category_switch_count',
            'switch_rate_per_interaction',
            'avg_gap_between_interactions_seconds_log1p',
        ]
        if c in out.columns
    ]
    keep_core = [
        c for c in [
            'user_id',
            'date',
            'num_unique_apps',
            'num_unique_categories',
            'total_duration',
            'avg_session_duration',
            'day_of_week_num',
            'is_weekend',
        ]
        if c in out.columns
    ]

    selected_cols = keep_core + ratio_cols + entropy_cols + switch_cols
    selected_cols = list(dict.fromkeys(selected_cols))
    cluster_df = out[selected_cols].copy()

    # Drop all *_duration except total_duration and avg_session_duration.
    drop_duration_cols = [
        c for c in cluster_df.columns
        if c.endswith('_duration') and c not in {'total_duration', 'avg_session_duration'}
    ]
    cluster_df = cluster_df.drop(columns=drop_duration_cols, errors='ignore')

    # Final NA handling for clustering matrix safety.
    numeric_cols = cluster_df.select_dtypes(include=[np.number]).columns.tolist()
    cluster_df[numeric_cols] = cluster_df[numeric_cols].fillna(0.0)

    return cluster_df


# 1) Sparsity check + merge sparse sub-categories.
user_day_features_post, sparsity_report = merge_sparse_category_ratio_features(user_day_features, zero_threshold=0.95)

print('=== Sparsity Check (95% zeros) ===')
if len(sparsity_report) == 0:
    print('No sparse sub-category ratio columns were merged.')
else:
    print('Merged sparse ratio columns:')
    print(sparsity_report.to_string(index=False))

# 2) Build clustering-ready features with your keep/drop rules.
clustering_features = build_clustering_features(user_day_features_post)

print('\n=== Clustering Feature Table ===')
print('Shape:', clustering_features.shape)
print('Columns:')
print(clustering_features.columns.tolist())
print('\nMissing per column:')
print(clustering_features.isna().sum())

# 3) Save clustering feature table.
clustering_out = project_root / 'data' / 'features' / 'lsapp_user_day_clustering_features.tsv'
clustering_features.to_csv(clustering_out, sep='\t', index=False)
print(f'\nSaved clustering feature table: {clustering_out}')

=== Sparsity Check (95% zeros) ===
Merged sparse ratio columns:
                sparse_col     parent_col  zero_ratio           action
       social_dating_ratio   social_ratio    0.992776 merged_to_parent
    social_knowledge_ratio   social_ratio    0.991400 merged_to_parent
       rewards_video_ratio  rewards_ratio    0.990024 merged_to_parent
       rewards_games_ratio  rewards_ratio    0.987272 merged_to_parent
         news_search_ratio     news_ratio    0.984176 merged_to_parent
      health_fitness_ratio   health_ratio    0.983144 merged_to_parent
      survey_rewards_ratio   survey_ratio    0.982800 merged_to_parent
        news_utility_ratio     news_ratio    0.972480 merged_to_parent
    shopping_rewards_ratio shopping_ratio    0.970416 merged_to_parent
shopping_marketplace_ratio shopping_ratio    0.962504 merged_to_parent

=== Clustering Feature Table ===
Shape: (2907, 42)
Columns:
['user_id', 'date', 'num_unique_apps', 'num_unique_categories', 'total_duration', 'avg_session